# Expanded-Dataset Autoencoder: Flattened Swath Layout

Curated from the archived research notebook `new UCSD-1D.ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

The original workflow monitors arrays named `testing` during training.
Its displayed validation curves are not an independent final test. Training
is disabled until `ALLOW_TRAINING` is explicitly enabled.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='models/expanded_flattened_autoencoder')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr


In [ ]:
file_path = 'new_ssh_training_data.nc'
ds = xr.open_dataset(input_path(file_path))
training = ds["ssh_training_data"].values
training_target = ds["ssh_training_target"].values
print(training.shape)

# Array values are loaded above; release the NetCDF file handle.
ds.close()


In [ ]:
file_path = 'new_ssh_testing_data.nc'
ds = xr.open_dataset(input_path(file_path))
testing = ds["ssh_testing_data"].values
testing_target =ds["ssh_testing_target"].values
print(testing.shape)

# Array values are loaded above; release the NetCDF file handle.
ds.close()


In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Split the data into training and testing sets
training = np.transpose(training, (2, 1, 0))
training_target = np.transpose(training_target, (2, 1, 0))
testing = np.transpose(testing, (2, 1, 0))
testing_target = np.transpose(testing_target, (2, 1, 0))
# Convolution layers require an explicit trailing channel axis.
training = training.astype(np.float32)[..., np.newaxis]
training_target = training_target.astype(np.float32)[..., np.newaxis]
testing = testing.astype(np.float32)[..., np.newaxis]
testing_target = testing_target.astype(np.float32)[..., np.newaxis]
train, validation, train_target, validation_target = train_test_split(training, training_target, test_size=0.1, random_state=42)

# Check data types and shapes
print(f'x_train shape: {train.shape}, dtype: {train.dtype}')
print(f'y_train shape: {train_target.shape}, dtype: {train_target.dtype}')
print(f'x_test shape: {validation.shape}, dtype: {validation.dtype}')
print(f'y_test shape: {validation_target.shape}, dtype: {validation_target.dtype}')


In [ ]:
# Set the hyperparameters
learning_rate = 1e-4 # Adjust this value as needed
optimizer = Adam(learning_rate=learning_rate)
Epoch = 100000
# early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_mse', patience=10000, verbose=0, mode='min', restore_best_weights=True)


In [ ]:
import numpy as np

# Define the model
input_shape = (train.shape[1], train.shape[2], 1)  # (time, points, channels)
input_img = Input(shape=input_shape)

# Encoder
x = Conv2D(32, (10, 10), padding='same')(input_img)  # Reduced filters
encoded = Conv2D(64, (10, 10), padding='same')(x)  # Reduced filters

# Decoder
x = Conv2DTranspose(64, (10, 10), padding='same')(encoded)  # Use Conv2DTranspose
x = Conv2DTranspose(32, (10, 10), padding='same')(x)  # Use Conv2DTranspose
decoded = Conv2D(1, (10, 10), activation='tanh', padding='same')(x)

# Combine encoder and decoder into an autoencoder model
autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer=optimizer, loss='mean_absolute_error', metrics=['mae'])
print(autoencoder.summary())


In [ ]:
print(tf.keras.losses.MeanAbsoluteError()(training , training_target))
print(tf.keras.losses.MeanAbsoluteError()(testing , testing_target))


In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError('Set ALLOW_TRAINING=True after reviewing the epoch count and validation protocol.')

# Train the autoencoder
train_history = autoencoder.fit(train, train_target,
                epochs=Epoch,
                batch_size=32,
                shuffle=True,
                verbose=2,
                validation_data=(testing, testing_target))
                #callbacks=[early_stop]


In [ ]:
Training_History=np.zeros((4,Epoch))
Training_History[0,:] = np.array(train_history.history['loss'])
Training_History[1,:] = np.array(train_history.history['mae'])
Training_History[2,:] = np.array(train_history.history['val_loss'])
Training_History[3,:] = np.array(train_history.history['val_mae'])


In [ ]:
plt.plot(Training_History[1,:], color='blue', label='Training MAE')
plt.plot(Training_History[3,:], color='red', label='Monitored validation MAE')
plt.legend()
plt.xlabel('epochs')
plt.ylabel('MAE')
plt.ylim(0,0.01)
plt.show()


In [ ]:
# Save the entire model
autoencoder.save(output_path('autoencoder.keras'))


In [ ]:
denoised_ssha = autoencoder.predict(testing)
# Reshape the denoised images back to the original shape without the channel dimension
# Keep channel axes aligned for error evaluation.
print(tf.keras.losses.MeanAbsoluteError()(testing , testing_target))
print(tf.keras.losses.MeanAbsoluteError()(denoised_ssha, testing_target))
